In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

import sqlite3
import requests


ji


In [ ]:
load_dotenv(dotenv_path=".env", override=True)

llm = ChatOpenAI(model_name="meta-llama/Llama-3.1-70B-Instruct",temperature=0)


def create_db_engine():
    with open("utils/generate_data.db", "rb") as f:
        db_bytes = f.read()

    # Step 2 — Load it into in-memory SQLite
    in_memory_conn = sqlite3.connect(":memory:", check_same_thread=False)
    disk_conn = sqlite3.connect("utils/generate_data.db", check_same_thread=False)
    disk_conn.backup(in_memory_conn)  # copies all data from file into memory
    disk_conn.close()

engine = create_engine("sqlite:///utils/generate_data.db", creator = lambda: sqlite3.connect("utils/generate_data.db"), poolclass=StaticPool,
                        connect_args={"check_same_thread":False})

db= SQLDatabase(engine)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver # short-term memory
from langgraph.store.memory import InMemoryStore #persistent store

in_memory_store = InMemoryStore()

checkpointer = MemorySaver()

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated,List
from langgraph.graph.message import AnyMessage,add_messages
from langgraph.managed.is_last_step import RemainingSteps


class State(TypedDict):
    customer_id:str
    messages: Annotated[list[AnyMessage],add_messages]
    loaded_memory:str
    remaining_steps: RemainingSteps

In [ ]:
from langchain_core.tools import tool
import ast

@tool
def get_transactions_by_account(customer: str):
    """Get Transaction by a customer"""

    return db.run(
        f"""SELECT CUSTOMER.CustomerId, CUSTOMER.FirstName FROM Customers JOIN Accounts ON Customers.CustomerId = AccountsCustomerId
        WHERE Customers.FirstName LIKE '%{customer}%'
        """,
        include_columns=True
    )

@tool
def get_loan_details_by_customer(customer:str):
    """Get Loan Details of Customer"""
    return db.run(
        """
        
        """
    )

@tool
def get_products_by_category